<style>
  /* 1. Fix standard input code cells */
  pre, code, .jp-CodeCell .jp-Cell-inputWrapper {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
  }

  /* 2. Fix display() outputs and rendered Markdown */
  .jp-OutputArea-output, 
  .jp-OutputArea-output pre, 
  .jp-RenderedHTMLCommon pre, 
  .jp-RenderedMarkdown pre {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
      
      /* Forces the browser to print the overflow instead of hiding it in a scrollbox */
      overflow-x: visible !important; 
      max-width: 100% !important;
  }
</style>

In [1]:
%load_ext autoreload
%autoreload 2

### **Prediksi *Pavement Condition Index* menggunakan Model *Machine Learning (RF, XGBoost, ANN)***

#### 0. Konfigurasi parameter eksperimen

In [2]:
# Nama file dataset
DATASET = "pci-ml-26.xlsx"

# Parameter split data
SPLIT_PARAMS = {
    "n_seed": 20000,
    "test_size": 0.2,
    "share_tol": 0.01,
    "pci_min": 0,
}

# Cross Validation
CV_SPLITS = 5

# Parameter tuning Random Forest
RF_N_ITER = 100

# Parameter tuning XGBoost


# Parameter tuning ANN


# Model seed (RF, XGBoost, ANN)
SEED = 42

# Memori
from pci_ml.memory import configure
configure(release=True)


#### 1. Pembersihan data dan rekayasa fitur

In [3]:
# Memuat dataset, membersihkan data, dan menambahkan fitur agregat
from IPython.display import HTML
from pci_ml.data import load_data, clean_data, to_density, engineer_features

from pci_ml.ui import df_preview

# muat dataset mentah
raw = load_data(DATASET)
# membersihkan data duplikat, data tidak valid
df, features = clean_data(raw)
# menghitung fitur density
df, features = to_density(df, features)
# Feature engineering: total densitas, densitas per tingkat keparahan (L/M/H)
df, features = engineer_features(df, features)

print(f"File dimuat: {DATASET} | Data telah dibersihkan | {len(features)} fitur telah dihitung.")
display(HTML(df_preview(df)))

File dimuat: pci-ml-26.xlsx | Data telah dibersihkan | 59 fitur telah dihitung.


ROAD_NAME,ROAD_ID,SEGMENT_ID,GATOR_CRACK_L,GATOR_CRACK_M,GATOR_CRACK_H,…,PCI
Guru Bangkol,R001,S001,0,0,0,…,85.5
Guru Bangkol,R001,S002,0,0.0039,0,…,73.34
Guru Bangkol,R001,S003,0,0,0,…,89.94
Guru Bangkol,R001,S004,0.0014,0.0231,0,…,73.63
Guru Bangkol,R001,S005,0,0,0,…,96.19
Guru Bangkol,R001,S006,0,0,0,…,84.01
Guru Bangkol,R001,S007,0,0,0,…,90.7
Guru Bangkol,R001,S008,0.0243,0,0,…,73.72
Guru Bangkol,R001,S009,0,0,0.0044,…,77.9
Guru Bangkol,R001,S010,0,0,0,…,84.22


#### 2. Pemisahan data latih dan data uji

In [4]:
# Memisah data sesuai kriteria:
# Kriteria 1: test, train = 80%, 20% (dengan toleransi 1%)
# Kriteria 2: tidak ada fitur yang hanya muncul di test
# Kriteria 3: nilai target ekstrim, test harus punya jalan dengan PCI = 0
from pci_ml.data import DataSplit
from pci_ml.memory import release

print(f"Jumlah seed dicari: {SPLIT_PARAMS["n_seed"]} | "
      f"Test size: {SPLIT_PARAMS["test_size"]} | "
      f"Toleransi distribusi: {SPLIT_PARAMS["share_tol"]} | "
      f"PCI minimum: {SPLIT_PARAMS["pci_min"]}")

result = DataSplit(df, features, DATASET, SPLIT_PARAMS).run()
print(result["summary"])

release("result", "df")


Jumlah seed dicari: 20000 | Test size: 0.2 | Toleransi distribusi: 0.01 | PCI minimum: 0
Seed used: 16650 (Gap: 0.2172)
Train: 935 rows (79.44%) | Test: 242 rows (20.56%)
Roads: 27 train | 7 test
Saved to: "C:\Users\gearb\Desktop\pci-ml-single\pci-ml-rf\pci-ml\datasets\pci-ml-26\n20000_t0.2_s0.01_p0"


#### 3. Persiapan input model *machine learning*

In [5]:
# Mempersiapkan input model
import numpy as np
from IPython.display import Markdown
from pci_ml.data import ModelInput, PCI, ROAD_ID, METADATA, ENGINEERED_FEATURES
from pci_ml.memory import release

# Muat file dataset yang sudah dipisah (train/test, features, dan metadata)
mi = ModelInput(DATASET)
tag = mi.find(SPLIT_PARAMS)
train_df, test_df, features, meta = mi.load(tag)

# definisi dataframe ke ndarray untuk kebutuhan model
X_train = train_df[features].to_numpy(dtype="float32")
X_test = test_df[features].to_numpy(dtype="float32")
y_train = train_df[PCI].to_numpy(dtype="float32")    # PCI scale 0-100; divide by 100 in the ANN cell
y_test = test_df[PCI].to_numpy(dtype="float32")

groups_train = train_df[ROAD_ID].to_numpy()
test_ids = test_df[[ROAD_ID, *METADATA]].reset_index(drop=True)

# Summary
n_tr, n_te = len(X_train), len(X_test)
total = n_tr + n_te
split_text = (f"**Train:** {X_train.shape} ({n_tr / total:.2%}) | "
              f"**Test:** {X_test.shape} ({n_te / total:.2%})")
display(Markdown(f"**Split terpilih:** `{tag}`\n\n{split_text}"))

**Split terpilih:** `n20000_t0.2_s0.01_p0`

**Train:** (935, 59) (79.44%) | **Test:** (242, 59) (20.56%)

#### 4. Pelatihan dan pengujian model *Random Forest*

In [6]:
# Tuning hyperparameter dan pelatihan model Random Forest final
from IPython.display import HTML, Markdown, display
from pci_ml.experiments import evaluate_oof, evaluate_predictions
from pci_ml.random_forest import tune_random_forest
from pci_ml.ui import build_hyperparameter_table, format_table

RF_SEARCH_SPACE = {
    "n_estimators": [300],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2],
    "max_features": [0.1, 0.15, 0.2, 0.25, 0.3, 0.4],
    "max_samples": [0.5, 0.6, 0.7, 0.8, 0.9],
}

print(f"Tuning Random Forest: {RF_N_ITER} iterasi | {CV_SPLITS} fold CV")
rf_result = tune_random_forest(
    X_train, y_train, groups_train,
    RF_SEARCH_SPACE,
    n_iter=RF_N_ITER,
    cv_splits=CV_SPLITS,
    random_state=SEED,
)

rf_best_params = rf_result["params"]
tuned_model = rf_result["model"]
rf_cv = evaluate_oof(
    y_train, rf_result["oof"], groups_train, rf_result["folds"]
)["pooled"]
tuned_test = evaluate_predictions(y_test, tuned_model.predict(X_test))

param_explanations = {
    "n_estimators": "Jumlah pohon keputusan.",
    "max_depth": "Kedalaman maksimum tiap pohon.",
    "min_samples_split": "Jumlah sampel minimum agar sebuah node boleh dipecah.",
    "min_samples_leaf": "Jumlah sampel minimum di tiap daun.",
    "max_features": "Fraksi fitur yang dipertimbangkan di tiap pohon.",
    "max_samples": "Fraksi data latih yang dipakai untuk tiap pohon.",
}

param_table = build_hyperparameter_table(
    rf_best_params, RF_SEARCH_SPACE.keys(), param_explanations
)

evaluation_table = format_table(
    ["Evaluasi", "MAE", "RMSE", "R²"],
    [
        ["Cross-Validation", f"{rf_cv['mae']:.4f}", f"{rf_cv['rmse']:.4f}", f"{rf_cv['r2']:.4f}"],
        ["Test", f"{tuned_test['mae']:.4f}", f"{tuned_test['rmse']:.4f}", f"{tuned_test['r2']:.4f}"],
    ],
    aligns=["left", "center", "center", "center"],
)

n_test_roads = test_ids["ROAD_ID"].nunique()

display(Markdown("### Hyperparameter terbaik"))
display(HTML(param_table))
display(Markdown("### Hasil evaluasi"))
display(HTML(evaluation_table))


Tuning Random Forest: 100 iterasi | 5 fold CV


### Hyperparameter terbaik

Hyperparameter,Nilai,Keterangan
n_estimators,300,Jumlah pohon keputusan.
max_depth,20,Kedalaman maksimum tiap pohon.
min_samples_split,4,Jumlah sampel minimum agar sebuah node boleh dipecah.
min_samples_leaf,1,Jumlah sampel minimum di tiap daun.
max_features,0.3,Fraksi fitur yang dipertimbangkan di tiap pohon.
max_samples,0.9,Fraksi data latih yang dipakai untuk tiap pohon.


### Hasil evaluasi

Evaluasi,MAE,RMSE,R²
Cross-Validation,5.8378,8.8181,0.8502
Test,5.7101,8.8617,0.8465
